In [1]:
import math
import re
from pathlib import Path
from typing import Optional

import numpy as np
import napari
import tifffile
from liffile import LifFile
from magicgui import magicgui
from magicgui.widgets import Container, TextEdit
from scipy.ndimage import binary_dilation, rotate
from skimage import util
from skimage.draw import line
from skimage.filters import median, sobel, threshold_triangle
from skimage.measure import label, regionprops, regionprops_table
from skimage.morphology import disk, erosion, remove_small_objects
from skimage.transform import ProjectiveTransform, resize, warp, probabilistic_hough_line


In [2]:
# Minimal working GUI: load → segment/view → XY crop → Z crop → (optional) rescale to 5µm

def _is_valid_um(v) -> bool:
    if v is None:
        return False
    try:
        vf = float(v)
    except Exception:
        return False
    return bool(np.isfinite(vf) and vf > 0)


def _unit_to_um_factor(unit: str) -> Optional[float]:
    if not unit:
        return None
    u = unit.strip().lower()
    if u in ("um", "µm", "micron", "microns", "micrometer", "micrometers"):
        return 1.0
    if u in ("nm", "nanometer", "nanometers"):
        return 1e-3
    if u in ("mm", "millimeter", "millimeters"):
        return 1e3
    if u in ("m", "meter", "meters"):
        return 1e6
    return None


def _parse_ome_physical_sizes_um(ome_xml: str) -> tuple[Optional[float], Optional[float], Optional[float]]:
    if not ome_xml:
        return None, None, None
    def _attr(name: str):
        m = re.search(rf'{name}="([^"]+)"', ome_xml)
        return m.group(1) if m else None
    def _unit(name: str):
        m = re.search(rf'{name}Unit="([^"]+)"', ome_xml)
        return m.group(1) if m else None

    px = _attr("PhysicalSizeX")
    py = _attr("PhysicalSizeY")
    pz = _attr("PhysicalSizeZ")
    ux = _unit("PhysicalSizeX") or _unit("PhysicalSizeY") or _unit("PhysicalSizeZ")
    f = _unit_to_um_factor(ux) if ux else None
    def _conv(val):
        if val is None:
            return None
        try:
            vv = float(val)
        except Exception:
            return None
        if f is None:
            # OME defaults are usually µm if unspecified, but don't guess if missing
            return None
        return vv * f

    return _conv(px), _conv(py), _conv(pz)


def read_tiff_voxel_um(path: Path) -> tuple[Optional[float], Optional[float]]:
    """Return (z_step_um, xy_step_um) if present; otherwise (None, None)."""
    p = Path(path)
    if not p.exists():
        return None, None

    z_um = None
    xy_um = None

    with tifffile.TiffFile(str(p)) as tif:
        # 1) OME-TIFF (best case)
        try:
            ome_xml = tif.ome_metadata
        except Exception:
            ome_xml = None
        if ome_xml:
            x_um, y_um, z_um_ome = _parse_ome_physical_sizes_um(ome_xml)
            if _is_valid_um(x_um) and _is_valid_um(y_um):
                xy_um = float((float(x_um) + float(y_um)) / 2.0)
            if _is_valid_um(z_um_ome):
                z_um = float(z_um_ome)

        # 2) ImageJ metadata (spacing, unit)
        try:
            ij = tif.imagej_metadata if getattr(tif, "is_imagej", False) else None
        except Exception:
            ij = None
        if ij:
            unit = str(ij.get("unit") or ij.get("Unit") or "")
            f = _unit_to_um_factor(unit)
            spacing = ij.get("spacing")
            if spacing is not None and f is not None:
                try:
                    z_um = float(spacing) * float(f)
                except Exception:
                    pass

        # 3) Resolution tags for XY
        if xy_um is None:
            try:
                page0 = tif.pages[0]
                tags = page0.tags
                xres = tags.get("XResolution")
                yres = tags.get("YResolution")
                runit = tags.get("ResolutionUnit")
                if xres is not None and yres is not None:
                    x_num, x_den = xres.value
                    y_num, y_den = yres.value
                    px_per_unit_x = float(x_num) / float(x_den)
                    px_per_unit_y = float(y_num) / float(y_den)
                    unit_code = int(runit.value) if runit is not None else 1
                    # TIFF: 2=inches, 3=centimeters, 1=none/unknown
                    if unit_code == 2:
                        um_per_unit = 25400.0
                    elif unit_code == 3:
                        um_per_unit = 10000.0
                    else:
                        um_per_unit = None

                    if um_per_unit is not None:
                        x_um = um_per_unit / px_per_unit_x if px_per_unit_x > 0 else None
                        y_um = um_per_unit / px_per_unit_y if px_per_unit_y > 0 else None
                        if _is_valid_um(x_um) and _is_valid_um(y_um):
                            xy_um = float((float(x_um) + float(y_um)) / 2.0)
                    else:
                        # last-ditch heuristic (matches your previous behavior)
                        x_um = 1.0 / px_per_unit_x if px_per_unit_x > 0 else None
                        y_um = 1.0 / px_per_unit_y if px_per_unit_y > 0 else None
                        if _is_valid_um(x_um) and _is_valid_um(y_um):
                            xy_um = float((float(x_um) + float(y_um)) / 2.0)
            except Exception:
                pass

        # 4) ImageDescription fallback for Z spacing
        if z_um is None:
            try:
                desc = tif.pages[0].tags.get("ImageDescription")
                desc_txt = str(desc.value) if desc is not None else ""
                m = re.search(r"spacing=([0-9eE+\-\.]+)", desc_txt)
                if m:
                    z_um = float(m.group(1))
            except Exception:
                pass

    if not _is_valid_um(z_um):
        z_um = None
    if not _is_valid_um(xy_um):
        xy_um = None
    return z_um, xy_um


def read_lif_voxel_um(lif_path: Path, image_index: int) -> tuple[Optional[float], Optional[float]]:
    """Return (z_step_um, xy_step_um) from LIF coords if available."""
    p = Path(lif_path)
    if not p.exists():
        return None, None
    try:
        with LifFile(p) as lif:
            img = lif.images[int(image_index)]
            # liffile provides xarray coordinates via .xarray() on many builds
            xa = None
            if hasattr(img, "xarray"):
                try:
                    xa = img.xarray()
                except Exception:
                    xa = None
            if xa is None or not hasattr(xa, "coords"):
                return None, None

            def _step(axis: str):
                if axis not in xa.coords or xa.coords[axis].size < 2:
                    return None
                try:
                    return float(xa.coords[axis][1] - xa.coords[axis][0])
                except Exception:
                    return None

            sx = _step("X")
            sy = _step("Y")
            sz = _step("Z")

            # lif coords are often meters; convert to µm if value looks small
            def _to_um(v):
                if v is None:
                    return None
                vv = float(v)
                if not np.isfinite(vv) or vv == 0:
                    return None
                # Heuristic: if step < 1e-3, treat as meters
                if abs(vv) < 1e-3:
                    return vv * 1e6
                return vv  # already in µm or something close

            x_um = _to_um(sx)
            y_um = _to_um(sy)
            z_um = _to_um(sz)
            xy_um = None
            if _is_valid_um(x_um) and _is_valid_um(y_um):
                xy_um = float((float(x_um) + float(y_um)) / 2.0)
            if not _is_valid_um(z_um):
                z_um = None
            return z_um, xy_um
    except Exception:
        return None, None


def _to_gray_2d(arr: np.ndarray) -> np.ndarray:
    a = np.asarray(arr)
    if a.ndim == 2:
        return a
    if a.ndim == 3:
        return np.mean(a, axis=-1)
    raise ValueError(f"Expected 2D or 3D array; got shape={a.shape}")


def _focus_score_2d(img2d: np.ndarray) -> float:
    g = _to_gray_2d(img2d).astype(np.float32, copy=False)
    return float(np.mean(np.abs(sobel(g))))


def best_focus_slice(stack: np.ndarray, patch: int = 50, n_sampling: int = 20) -> tuple[int, list[int]]:
    """Return (global_z0, sampled_best_zs). stack is (Z,Y,X[,C])."""
    st = np.asarray(stack)
    if st.ndim == 4:
        st_score = np.mean(st, axis=-1)
    else:
        st_score = st
    if st_score.ndim != 3:
        raise ValueError(f"Expected ZYX or ZYXC stack; got shape={st.shape}")

    nz, ny, nx = st_score.shape
    # global score
    scores = [ _focus_score_2d(st_score[z]) for z in range(nz) ]
    global_z0 = int(np.argmax(scores))

    # sampled patch scores
    zs = []
    if n_sampling <= 0:
        return global_z0, zs
    rng = np.random.default_rng(0)
    half = max(1, int(patch) // 2)
    ys = rng.integers(low=half, high=max(half + 1, ny - half), size=int(n_sampling))
    xs = rng.integers(low=half, high=max(half + 1, nx - half), size=int(n_sampling))
    for y, x in zip(ys, xs):
        y0, y1 = int(max(0, y - half)), int(min(ny, y + half))
        x0, x1 = int(max(0, x - half)), int(min(nx, x + half))
        patch_scores = [ _focus_score_2d(st_score[z, y0:y1, x0:x1]) for z in range(nz) ]
        zs.append(int(np.argmax(patch_scores)))
    return global_z0, zs


class DeviceSegmentationApp:
    def __init__(self):
        self.viewer = napari.Viewer()

        self._selected_image_folder: Optional[Path] = None
        self._selected_lif: Optional[Path] = None
        self._image_paths: list[Path] = []
        self._image_choice_map: dict[str, int] = {}

        self._roi_layer = None
        self._cropped_layer = None

        self._last_image = None
        self._last_image_path = None
        self._last_stack = None

        self._last_focus_downsample = 1
        self._last_focus_n_sampling = 20
        self._last_focus_patch = 50

        # voxel metadata (cached + snapshot)
        self._voxel_cache: dict[tuple, tuple[Optional[float], Optional[float]]] = {}
        self._source_z_step_um: Optional[float] = None
        self._source_xy_step_um: Optional[float] = None
        self._active_voxel_key = None
        self._last_z_step_um: Optional[float] = None
        self._last_xy_step_um: Optional[float] = None

        # focus stats used for Z crop bounds + warnings
        self._last_global_z0: Optional[int] = None
        self._last_sampled_best_z_min: Optional[int] = None
        self._last_sampled_best_z_max: Optional[int] = None
        self._last_nz: Optional[int] = None

        # crop outputs
        self._cropped_stack_xy_raw = None
        self._cropped_stack_z_raw = None
        self._cropped_stack_5um_raw = None

        self._updating_z_widgets = False

        self.images_output = TextEdit(value="")
        try:
            self.images_output.native.setReadOnly(True)
        except Exception:
            pass
        self.images_output.min_height = 120
        self.images_output.max_height = 300

        self.voxel_output = TextEdit(value="Voxel metadata (µm): z_step=?, xy_step=?")
        try:
            self.voxel_output.native.setReadOnly(True)
        except Exception:
            pass
        self.voxel_output.min_height = 55
        self.voxel_output.max_height = 90

        @magicgui(image_source={"label": "Folder/.tif/.lif", "mode": "r"}, call_button="Load images")
        def list_images(image_source: Path = Path()):
            self._list_images(image_source)

        @magicgui(
            image_choice={"label": "Image", "choices": ["(load images)"], "widget_type": "ComboBox"},
            focus_downsample={"label": "Downsample", "min": 1, "max": 16, "step": 1},
            focus_n_sampling={"label": "n_sampling", "min": 0, "max": 200, "step": 1},
            focus_patch={"label": "patch", "min": 5, "max": 512, "step": 1},
            mask_central_region={"label": "Mask central region"},
            see_interim_layers={"label": "See interim layers (debug)"},
            clear_layers={"label": "Clear viewer first"},
            call_button="Segment + View",
        )
        def segment_and_view(
            image_choice: str = "(load images)",
            focus_downsample: int = 4,
            focus_n_sampling: int = 20,
            focus_patch: int = 50,
            mask_central_region: bool = False,
            see_interim_layers: bool = True,
            clear_layers: bool = False,
        ):
            self._segment_and_view(
                image_choice=image_choice,
                focus_downsample=focus_downsample,
                focus_n_sampling=focus_n_sampling,
                focus_patch=focus_patch,
                mask_central_region=mask_central_region,
                see_interim_layers=see_interim_layers,
                clear_layers=clear_layers,
            )

        @magicgui(call_button="Create cropped aligned")
        def apply_crop():
            self._apply_crop_from_roi()

        @magicgui(
            z_range_um={"label": "Z range (um)", "min": 1.0, "max": 100000.0, "step": 1.0},
            z_min={"label": "z_min (slice)", "widget_type": "SpinBox", "min": 0, "step": 1},
            z_max={"label": "z_max (slice)", "widget_type": "SpinBox", "min": 0, "step": 1},
            call_button="Crop Z range",
        )
        def crop_z(z_range_um: float = 200.0, z_min: int = 0, z_max: int = 0):
            self._apply_z_crop(z_range_um=z_range_um)

        @magicgui(call_button="Rescale to 5×5×5 µm")
        def rescale_5um():
            self._apply_rescale_to_5um()

        self.list_images = list_images
        self.segment_and_view = segment_and_view
        self.apply_crop = apply_crop
        self.crop_z = crop_z
        self.rescale_5um = rescale_5um

        self.main_panel = Container(widgets=[
            self.list_images,
            self.segment_and_view,
            self.apply_crop,
            self.crop_z,
            self.rescale_5um,
            self.voxel_output,
            self.images_output,
        ])
        self.viewer.window.add_dock_widget(self.main_panel, area="right")

        self.segment_and_view.image_choice.changed.connect(self._update_segment_button)
        self.segment_and_view.image_choice.changed.connect(self._update_voxel_metadata_from_selection)
        self.segment_and_view.call_button.enabled = False
        self.crop_z.call_button.enabled = False
        self.rescale_5um.call_button.enabled = False
        self._update_segment_button()
        self._set_z_controls_enabled(False, reset_values=True)

        try:
            self.crop_z.z_range_um.changed.connect(lambda *_: self._update_z_crop_bounds(show_warning=True))
        except Exception:
            pass
        try:
            self.crop_z.z_min.changed.connect(lambda *_: self._update_z_range_from_slices(show_warning=True))
            self.crop_z.z_max.changed.connect(lambda *_: self._update_z_range_from_slices(show_warning=True))
        except Exception:
            pass

    # ---- voxel metadata ----
    def _voxel_key_from_selection(self):
        try:
            image_value = self.segment_and_view.image_choice.value
        except Exception:
            image_value = None
        image_index = self._image_choice_map.get(image_value) if image_value else None
        if image_index is None:
            return None
        if self._selected_lif is not None and Path(self._selected_lif).exists():
            return ("lif", str(Path(self._selected_lif).resolve()), int(image_index))
        if self._image_paths and 0 <= int(image_index) < len(self._image_paths):
            return ("tiff", str(Path(self._image_paths[int(image_index)]).resolve()))
        return None

    def _read_voxel_for_key(self, key):
        if key is None:
            return None, None
        if key[0] == "lif":
            _, lif_path, idx = key
            return read_lif_voxel_um(Path(lif_path), int(idx))
        if key[0] == "tiff":
            _, tif_path = key
            return read_tiff_voxel_um(Path(tif_path))
        return None, None

    def _update_voxel_metadata_from_selection(self, *_):
        key = self._voxel_key_from_selection()
        if key is None:
            # don't clobber snapshot; just show unknown
            self._last_z_step_um = None
            self._last_xy_step_um = None
            self._set_voxel_output_text()
            return
        if key not in self._voxel_cache:
            z, xy = self._read_voxel_for_key(key)
            self._voxel_cache[key] = (z, xy)
        z, xy = self._voxel_cache.get(key, (None, None))
        if _is_valid_um(z) and _is_valid_um(xy):
            self._last_z_step_um = float(z)
            self._last_xy_step_um = float(xy)
            self._source_z_step_um = float(z)
            self._source_xy_step_um = float(xy)
        else:
            self._last_z_step_um = None
            self._last_xy_step_um = None
        self._set_voxel_output_text()

    def get_voxel_size_um(self):
        # prefer frozen key (segmentation moment)
        key = self._active_voxel_key
        if key is not None:
            z, xy = self._voxel_cache.get(key, (None, None))
            if _is_valid_um(z) and _is_valid_um(xy):
                return (float(z), float(xy), float(xy))
        if _is_valid_um(self._source_z_step_um) and _is_valid_um(self._source_xy_step_um):
            z = float(self._source_z_step_um)
            xy = float(self._source_xy_step_um)
            return (z, xy, xy)
        if _is_valid_um(self._last_z_step_um) and _is_valid_um(self._last_xy_step_um):
            z = float(self._last_z_step_um)
            xy = float(self._last_xy_step_um)
            return (z, xy, xy)
        return (None, None, None)

    def _set_voxel_output_text(self):
        def _fmt(v):
            if not _is_valid_um(v):
                return "?"
            return f"{float(v):.4g}"
        z, y, _ = self.get_voxel_size_um()
        z_txt = _fmt(z)
        xy_txt = _fmt(y)
        if z_txt != "?" and xy_txt != "?":
            self.voxel_output.value = f"Voxel metadata (µm): (z,y,x)=({z_txt}, {xy_txt}, {xy_txt})"
        else:
            self.voxel_output.value = f"Voxel metadata (µm): z_step={z_txt}, xy_step={xy_txt}"

    # ---- UI helpers ----
    def _update_segment_button(self, *_):
        image_value = self.segment_and_view.image_choice.value
        image_ok = bool(self._image_choice_map) and image_value in self._image_choice_map
        self.segment_and_view.call_button.enabled = image_ok

    def _set_z_controls_enabled(self, enabled: bool, reset_values: bool = False):
        try:
            self.crop_z.z_range_um.enabled = bool(enabled)
            self.crop_z.z_min.enabled = bool(enabled)
            self.crop_z.z_max.enabled = bool(enabled)
            if reset_values:
                self._updating_z_widgets = True
                self.crop_z.z_range_um.value = 200.0
                self.crop_z.z_min.value = 0
                self.crop_z.z_max.value = 0
        except Exception:
            pass
        finally:
            self._updating_z_widgets = False

    def _sync_z_slice_widget_limits(self):
        try:
            nz = int(self._cropped_stack_xy_raw.shape[0]) if self._cropped_stack_xy_raw is not None else 0
            max_z = max(0, nz - 1)
            self.crop_z.z_min.min = 0
            self.crop_z.z_max.min = 0
            self.crop_z.z_min.max = max_z
            self.crop_z.z_max.max = max_z
        except Exception:
            pass

    # ---- IO ----
    def _list_images(self, image_source: Path):
        p = Path(image_source)
        if not p.exists():
            self.images_output.value = "[WARN] Please select a folder, a .tif/.tiff file, or a .lif file."
            return

        self._selected_image_folder = None
        self._selected_lif = None
        self._image_paths = []
        self._image_choice_map = {}
        choices = []
        choice_map = {}

        if p.is_dir():
            self._selected_image_folder = p
            image_files = sorted([q for q in p.iterdir() if q.is_file() and q.suffix.lower() in (".tif", ".tiff")])
            if not image_files:
                self.images_output.value = "[WARN] No .tif/.tiff images found in the selected folder."
                self.segment_and_view.image_choice.choices = ["(load images)"]
                self.segment_and_view.image_choice.value = "(load images)"
                self._update_segment_button()
                return
            self._image_paths = image_files
            for i, pth in enumerate(image_files):
                label = f"{i}: {pth.name}"
                choices.append(label)
                choice_map[label] = i
            self._image_choice_map = choice_map
            self.segment_and_view.image_choice.choices = choices
            self.segment_and_view.image_choice.value = choices[0]
            self._update_voxel_metadata_from_selection()
            self.images_output.value = f"[OK] Found {len(choices)} images in folder. Select one and click Segment + View."
            self._update_segment_button()
            return

        if p.suffix.lower() == ".lif":
            self._selected_lif = p
            try:
                with LifFile(p) as lif:
                    for i, img in enumerate(lif.images):
                        name = getattr(img, "name", None) or f"image_{i}"
                        label = f"{i}: {name}"
                        choices.append(label)
                        choice_map[label] = i
            except Exception:
                self.images_output.value = "[ERROR] Unable to read .lif contents."
                self.segment_and_view.image_choice.choices = ["(load images)"]
                self.segment_and_view.image_choice.value = "(load images)"
                self._update_segment_button()
                return
            if not choices:
                self.images_output.value = "[WARN] No readable images found inside the selected .lif file."
                self.segment_and_view.image_choice.choices = ["(load images)"]
                self.segment_and_view.image_choice.value = "(load images)"
                self._update_segment_button()
                return
            self._image_choice_map = choice_map
            self.segment_and_view.image_choice.choices = choices
            self.segment_and_view.image_choice.value = choices[0]
            self._update_voxel_metadata_from_selection()
            self.images_output.value = f"[OK] Found {len(choices)} images in .lif. Select one and click Segment + View."
            self._update_segment_button()
            return

        if p.suffix.lower() in (".tif", ".tiff"):
            self._image_paths = [p]
            label = f"0: {p.name}"
            self._image_choice_map = {label: 0}
            self.segment_and_view.image_choice.choices = [label]
            self.segment_and_view.image_choice.value = label
            self._update_voxel_metadata_from_selection()
            self.images_output.value = "[OK] Loaded single .tif file. Select it and click Segment + View."
            self._update_segment_button()
            return

        self.images_output.value = "[WARN] Unsupported selection. Choose a folder, a .tif/.tiff file, or a .lif file."
        self.segment_and_view.image_choice.choices = ["(load images)"]
        self.segment_and_view.image_choice.value = "(load images)"
        self._update_segment_button()

    # ---- segmentation ----
    def _load_selected_image(self, image_index: int) -> tuple[np.ndarray, Optional[np.ndarray]]:
        """Return (in_focus_plane_2d, stack_or_None)."""
        if self._selected_lif is not None and Path(self._selected_lif).exists():
            with LifFile(self._selected_lif) as lif:
                img = lif.images[int(image_index)]
                arr = np.asarray(img.asarray())
            self._last_image_path = self._selected_lif
        else:
            p = self._image_paths[int(image_index)]
            arr = np.asarray(tifffile.imread(str(p)))
            self._last_image_path = p

        if arr.ndim == 2:
            return arr.astype(np.float32), None
        if arr.ndim == 3:
            # assume stack ZYX
            stack = arr.astype(np.float32)
            z0, zs = best_focus_slice(stack, patch=self._last_focus_patch, n_sampling=self._last_focus_n_sampling)
            self._last_global_z0 = int(z0)
            self._last_nz = int(stack.shape[0])
            if zs:
                self._last_sampled_best_z_min = int(np.min(zs))
                self._last_sampled_best_z_max = int(np.max(zs))
            else:
                self._last_sampled_best_z_min = None
                self._last_sampled_best_z_max = None
            return stack[int(z0)], stack
        if arr.ndim == 4:
            # assume ZYXC or CYXZ etc. Minimal: treat last axis as channels if small
            if arr.shape[-1] in (1, 3, 4):
                stack = arr.astype(np.float32)
            else:
                # fallback: collapse to mean and treat as ZYX
                stack = np.mean(arr.astype(np.float32), axis=-1)
            z0, zs = best_focus_slice(stack, patch=self._last_focus_patch, n_sampling=self._last_focus_n_sampling)
            self._last_global_z0 = int(z0)
            self._last_nz = int(stack.shape[0])
            if zs:
                self._last_sampled_best_z_min = int(np.min(zs))
                self._last_sampled_best_z_max = int(np.max(zs))
            else:
                self._last_sampled_best_z_min = None
                self._last_sampled_best_z_max = None
            plane = np.mean(stack[int(z0)], axis=-1) if stack.ndim == 4 else stack[int(z0)]
            return plane, stack

        raise ValueError(f"Unsupported image shape: {arr.shape}")

    def _segment_and_view(self, image_choice: str, focus_downsample: int, focus_n_sampling: int, focus_patch: int, mask_central_region: bool, see_interim_layers: bool, clear_layers: bool):
        if not self._image_choice_map:
            self.images_output.value = "[WARN] Click 'Load images' first."
            return
        image_index = self._image_choice_map.get(image_choice)
        if image_index is None:
            self.images_output.value = "[WARN] Please select an image from the dropdown."
            return

        # store focus params + freeze voxel metadata at segmentation start
        self._last_focus_downsample = max(1, int(focus_downsample))
        self._last_focus_n_sampling = int(focus_n_sampling)
        self._last_focus_patch = int(focus_patch)
        self._update_voxel_metadata_from_selection()
        self._active_voxel_key = self._voxel_key_from_selection()

        self.images_output.value = f"[INFO] Segmenting image_index={image_index}..."
        try:
            in_focus_plane, stack = self._load_selected_image(int(image_index))
        except Exception as e:
            self.images_output.value = f"[ERROR] Load failed: {type(e).__name__}: {e}"
            return

        self._last_stack = stack
        self._last_image = in_focus_plane

        try:
            (in_focus_plane, organoid_region, final_corners, cropped_rotated, debug) = self._segment_from_plane(in_focus_plane, mask_central_region)
        except Exception as e:
            self.images_output.value = f"[ERROR] Segmentation failed: {type(e).__name__}: {e}"
            return

        if clear_layers:
            self.viewer.layers.clear()
            self._roi_layer = None
            self._cropped_layer = None

        self._set_voxel_output_text()
        self._cropped_stack_xy_raw = None
        self._cropped_stack_z_raw = None
        self._cropped_stack_5um_raw = None
        self.crop_z.call_button.enabled = False
        self.rescale_5um.call_button.enabled = False
        self._set_z_controls_enabled(False, reset_values=True)

        self._add_image_if_nonzero(in_focus_plane, name="original")
        if see_interim_layers:
            self._add_image_if_nonzero(debug.get("sobel"), name="sobel")
            self._add_image_if_nonzero(debug.get("binary"), name="binary")
            self._add_labels_if_nonzero(debug.get("labels_to_dilate"), name="labels_to_dilate")
            post = self._add_labels_if_nonzero(debug.get("post_dilation_mask"), name="post_dilation_mask")
            if post is not None:
                post.opacity = 0.4
            dev = self._add_labels_if_nonzero(debug.get("device_mask"), name="device_mask")
            if dev is not None:
                dev.opacity = 0.4
            if organoid_region is not None:
                org = self._add_labels_if_nonzero(organoid_region, name="organoid_region")
                if org is not None:
                    org.opacity = 0.4
            self._add_image_if_nonzero(debug.get("edges"), name="edges")
            self._add_image_if_nonzero(debug.get("reconstructed"), name="reconstructed")
            self._add_image_if_nonzero(debug.get("reconstructed_mask"), name="reconstructed_mask")

        force_roi = clear_layers or self._roi_layer is None or len(getattr(self._roi_layer, "data", [])) == 0
        self._set_roi_layer(final_corners, force=force_roi)
        self.images_output.value = "[OK] Segmentation complete. Adjust the rectangle and click 'Create cropped aligned'."

    def _segment_from_plane(self, in_focus_plane: np.ndarray, mask_central_region: bool):
        in_focus_plane = _to_gray_2d(in_focus_plane)
        med = median(in_focus_plane, footprint=disk(7))
        sob = sobel(med)
        thresh = threshold_triangle(sob)
        binary = sob > thresh

        # clear center block
        h, w = binary.shape
        h3, w3 = h // 3, w // 3
        binary[h3:2*h3, w3:2*w3] = 0

        organoid_region = None
        if mask_central_region:
            organoid_region = self._mask_out_organoid(in_focus_plane)
            binary[organoid_region] = 0

        labels = label(binary)
        data = regionprops_table(labels, binary, properties=("label", "area", "eccentricity"))
        condition = (data["area"] > 100) & (data["eccentricity"] > 0.5)
        labels_to_dilate = util.map_array(labels, data["label"], data["label"] * condition)
        dilated_output = np.zeros_like(labels, dtype=np.uint8)

        base_selem = np.zeros((31, 31), dtype=bool)
        base_selem[15, :] = 1
        pad = base_selem.shape[0] // 2
        for region in regionprops(labels_to_dilate):
            angle_to_rotate = self._signed_orientation(region)
            rotated_selem = rotate(base_selem.astype(float), angle=90 + angle_to_rotate, reshape=False, order=0) > 0.5
            minr, minc, maxr, maxc = region.bbox
            r0 = max(minr - pad, 0)
            r1 = min(maxr + pad, labels_to_dilate.shape[0])
            c0 = max(minc - pad, 0)
            c1 = min(maxc + pad, labels_to_dilate.shape[1])
            mask_roi = labels_to_dilate[r0:r1, c0:c1] == region.label
            dilated = binary_dilation(mask_roi, structure=rotated_selem)
            dilated_view = dilated_output[r0:r1, c0:c1]
            dilated_view[dilated] = 255

        post_dilation_mask = np.logical_or(dilated_output, binary)
        clean_labels = label(~post_dilation_mask)
        props = regionprops(clean_labels)
        largest_prop = max(props, key=lambda p: p.area)
        device_mask = clean_labels == largest_prop.label

        edges = reconstructed = reconstructed_mask = None
        corners, angle_rad, centroid_xy = self._oriented_rect_corners_crop_necks_and_flares(device_mask)
        flag = corners is None or self._corners_touch_border(corners, device_mask.shape, margin=5)
        if flag:
            edges = remove_small_objects(labels_to_dilate > 0)
            segs = probabilistic_hough_line(edges, line_length=400, line_gap=900, threshold=70)
            reconstructed = np.zeros_like(edges, dtype=bool)
            for (x0, y0), (x1, y1) in segs:
                rr, cc = line(y0, x0, y1, x1)
                reconstructed[rr, cc] = True
            reconstructed_mask = np.logical_or(reconstructed, post_dilation_mask)
            updated_clean_labels = label(~reconstructed_mask)
            props = regionprops(updated_clean_labels)
            largest_prop = max(props, key=lambda p: p.area)
            new_device_mask = updated_clean_labels == largest_prop.label
            corners, _, _ = self._oriented_rect_corners_crop_necks_and_flares(new_device_mask)

        cropped_rotated = self._crop_rectified_from_corners(in_focus_plane, corners)
        debug = {
            "sobel": sob.astype(np.float32, copy=False),
            "binary": binary.astype(np.uint8, copy=False),
            "labels_to_dilate": labels_to_dilate.astype(np.int32, copy=False),
            "post_dilation_mask": post_dilation_mask.astype(np.uint8, copy=False),
            "device_mask": device_mask.astype(np.uint8, copy=False),
            "edges": edges.astype(np.uint8, copy=False) if edges is not None else None,
            "reconstructed": reconstructed.astype(np.uint8, copy=False) if reconstructed is not None else None,
            "reconstructed_mask": reconstructed_mask.astype(np.uint8, copy=False) if reconstructed_mask is not None else None,
        }
        return in_focus_plane, organoid_region, corners, cropped_rotated, debug

    # ---- ROI + cropping ----
    def _set_roi_layer(self, corners_xy, force=False):
        if corners_xy is None:
            self.images_output.value = "[WARN] No corners found for ROI."
            return
        corners_yx = np.asarray(corners_xy)[:, ::-1]
        if self._roi_layer is None or self._roi_layer not in self.viewer.layers:
            self._roi_layer = self.viewer.add_shapes(name="geometry")
        if force or len(self._roi_layer.data) == 0:
            self._roi_layer.data = [corners_yx]
        self._roi_layer.shape_type = ["rectangle"]
        self._roi_layer.edge_color = "red"
        self._roi_layer.face_color = "transparent"
        self._roi_layer.editable = True
        self._roi_layer.mode = "select"

    def _order_corners_clockwise(self, corners_xy: np.ndarray):
        c = np.asarray(corners_xy, dtype=float)
        centroid = c.mean(axis=0)
        angles = np.arctan2(c[:, 1] - centroid[1], c[:, 0] - centroid[0])
        order = np.argsort(angles)
        c = c[order]
        start = np.argmin(c[:, 0] + c[:, 1])
        c = np.roll(c, -start, axis=0)
        return c

    def _crop_rectified_from_corners(self, in_focus_plane: np.ndarray, corners_xy: np.ndarray):
        if corners_xy is None:
            return None
        c = self._order_corners_clockwise(corners_xy)
        w1 = np.hypot(c[1, 0] - c[0, 0], c[1, 1] - c[0, 1])
        w2 = np.hypot(c[2, 0] - c[3, 0], c[2, 1] - c[3, 1])
        h1 = np.hypot(c[3, 0] - c[0, 0], c[3, 1] - c[0, 1])
        h2 = np.hypot(c[2, 0] - c[1, 0], c[2, 1] - c[1, 1])
        width = int(max(w1, w2))
        height = int(max(h1, h2))
        if width <= 1 or height <= 1:
            return None
        dst = np.array([[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]], dtype=float)
        tform = ProjectiveTransform()
        if not tform.estimate(dst, c):
            return None
        warped = warp(in_focus_plane, tform, output_shape=(height, width), order=1, preserve_range=True)
        return warped.astype(in_focus_plane.dtype)

    def _crop_rectified_stack_from_corners(self, stack: np.ndarray, corners_xy: np.ndarray):
        if corners_xy is None:
            return None
        c = self._order_corners_clockwise(corners_xy)
        w1 = np.hypot(c[1, 0] - c[0, 0], c[1, 1] - c[0, 1])
        w2 = np.hypot(c[2, 0] - c[3, 0], c[2, 1] - c[3, 1])
        h1 = np.hypot(c[3, 0] - c[0, 0], c[3, 1] - c[0, 1])
        h2 = np.hypot(c[2, 0] - c[1, 0], c[2, 1] - c[1, 1])
        width = int(max(w1, w2))
        height = int(max(h1, h2))
        if width <= 1 or height <= 1:
            return None
        dst = np.array([[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]], dtype=float)
        tform = ProjectiveTransform()
        if not tform.estimate(dst, c):
            return None
        st = np.asarray(stack)
        if st.ndim == 4:
            st_score = np.mean(st, axis=-1)
        else:
            st_score = st
        out = np.zeros((st_score.shape[0], height, width), dtype=st_score.dtype)
        for z in range(st_score.shape[0]):
            out[z] = warp(st_score[z], tform, output_shape=(height, width), order=1, preserve_range=True).astype(st_score.dtype)
        return out

    def _scale_to_uint8_view(self, arr):
        if arr is None:
            return None
        a = np.asarray(arr)
        if a.dtype == np.uint8:
            return a
        amin = float(np.nanmin(a))
        amax = float(np.nanmax(a))
        if not np.isfinite(amin) or not np.isfinite(amax) or amax <= amin:
            return None
        v = (a - amin) / (amax - amin)
        return (np.clip(v, 0, 1) * 255).astype(np.uint8)

    def _add_or_update_image_layer(self, layer, data, name):
        if layer is not None and layer in self.viewer.layers:
            layer.data = data
            return layer
        return self._add_image_if_nonzero(data, name=name)

    def _has_signal(self, arr) -> bool:
        return arr is not None and np.any(arr)

    def _add_image_if_nonzero(self, data, name, **kwargs):
        if self._has_signal(data):
            return self.viewer.add_image(data, name=name, **kwargs)
        return None

    def _add_labels_if_nonzero(self, data, name, **kwargs):
        if self._has_signal(data):
            return self.viewer.add_labels(data, name=name, **kwargs)
        return None

    def _apply_crop_from_roi(self):
        if self._roi_layer is None or len(self._roi_layer.data) == 0:
            self.images_output.value = "[WARN] Draw or adjust the rectangle first."
            return
        if self._last_image is None:
            self.images_output.value = "[WARN] Run 'Segment + View' first."
            return
        roi_yx = np.asarray(self._roi_layer.data[0])
        if roi_yx.shape[0] < 4:
            self.images_output.value = "[WARN] Rectangle needs 4 corners."
            return
        corners_xy = roi_yx[:, ::-1]

        if self._last_stack is not None:
            corners_full = corners_xy * float(max(1, int(self._last_focus_downsample)))
            cropped_stack = self._crop_rectified_stack_from_corners(self._last_stack, corners_full)
            if cropped_stack is None:
                self.images_output.value = "[WARN] Crop failed for the selected rectangle (stack)."
                return
            self._cropped_stack_xy_raw = cropped_stack
            self._cropped_stack_z_raw = None
            self._cropped_stack_5um_raw = None
            self.crop_z.call_button.enabled = True
            self.rescale_5um.call_button.enabled = True
            self._set_z_controls_enabled(True)
            self._sync_z_slice_widget_limits()

            display_stack = self._scale_to_uint8_view(cropped_stack)
            self._cropped_layer = self._add_or_update_image_layer(self._cropped_layer, display_stack, "cropped_rotated")

            # recompute focus stats on cropped XY for z-crop guidance
            try:
                z0, zs = best_focus_slice(cropped_stack, patch=self._last_focus_patch, n_sampling=self._last_focus_n_sampling)
                self._last_global_z0 = int(z0)
                self._last_nz = int(cropped_stack.shape[0])
                if zs:
                    self._last_sampled_best_z_min = int(np.min(zs))
                    self._last_sampled_best_z_max = int(np.max(zs))
                else:
                    self._last_sampled_best_z_min = None
                    self._last_sampled_best_z_max = None
            except Exception:
                pass

            self.images_output.value = f"[OK] Cropped aligned stack created. global_z0={self._last_global_z0}. Now choose a Z range and click Crop Z range."
            self._update_z_crop_bounds(show_warning=False)
            return

        cropped = self._crop_rectified_from_corners(self._last_image, corners_xy)
        if cropped is None:
            self.images_output.value = "[WARN] Crop failed for the selected rectangle."
            return
        display_cropped = self._scale_to_uint8_view(cropped)
        self._cropped_layer = self._add_or_update_image_layer(self._cropped_layer, display_cropped, "cropped_rotated")
        self.images_output.value = "[OK] Cropped aligned image created."

    # ---- Z crop ----
    def _get_active_stack_for_z(self):
        return self._cropped_stack_xy_raw

    def _compute_z_min_max_slices(self, z_range_um: float):
        try:
            z_range_um = float(z_range_um)
        except Exception:
            return None, None
        stack = self._get_active_stack_for_z()
        if stack is None:
            return None, None
        nz = int(stack.shape[0])
        self._last_nz = nz
        if self._last_global_z0 is None:
            return None, None
        if self._last_z_step_um is None or float(self._last_z_step_um) <= 0:
            return None, None
        z_range_px = max(1, int(round(z_range_um / float(self._last_z_step_um))))
        if z_range_px >= nz:
            return 0, nz - 1
        half = z_range_px // 2
        z0 = int(np.clip(int(self._last_global_z0), 0, nz - 1))
        lower = z0 - half
        upper = lower + (z_range_px - 1)
        if lower < 0:
            return 0, z_range_px - 1
        if upper > (nz - 1):
            return nz - z_range_px, nz - 1
        return int(lower), int(upper)

    def _update_z_crop_bounds(self, *_, show_warning: bool = False):
        if getattr(self, "_updating_z_widgets", False):
            return
        if self._cropped_stack_xy_raw is None:
            return
        try:
            z_range_um = float(self.crop_z.z_range_um.value)
        except Exception:
            return
        z_min, z_max = self._compute_z_min_max_slices(z_range_um)
        if z_min is None or z_max is None:
            return
        try:
            self._updating_z_widgets = True
            self.crop_z.z_min.value = int(z_min)
            self.crop_z.z_max.value = int(z_max)
            self.crop_z.z_min.max = max(0, int(self._cropped_stack_xy_raw.shape[0]) - 1)
            self.crop_z.z_max.max = max(0, int(self._cropped_stack_xy_raw.shape[0]) - 1)
        finally:
            self._updating_z_widgets = False
        if show_warning and self._last_sampled_best_z_min is not None:
            if (self._last_sampled_best_z_min < z_min) or (self._last_sampled_best_z_max > z_max):
                self.images_output.value = f"[WARN] slices {z_min}..{z_max} do NOT cover sampled best-zs ({self._last_sampled_best_z_min}..{self._last_sampled_best_z_max})."

    def _update_z_range_from_slices(self, *_, show_warning: bool = False):
        if getattr(self, "_updating_z_widgets", False):
            return
        if self._cropped_stack_xy_raw is None:
            return
        try:
            z_min = int(self.crop_z.z_min.value)
            z_max = int(self.crop_z.z_max.value)
        except Exception:
            return
        nz = int(self._cropped_stack_xy_raw.shape[0])
        z_min = int(np.clip(z_min, 0, nz - 1))
        z_max = int(np.clip(z_max, 0, nz - 1))
        if z_min > z_max:
            z_min, z_max = z_max, z_min
        try:
            self._updating_z_widgets = True
            self.crop_z.z_min.value = int(z_min)
            self.crop_z.z_max.value = int(z_max)
        finally:
            self._updating_z_widgets = False
        if self._last_z_step_um is not None and float(self._last_z_step_um) > 0:
            z_range_um = float((z_max - z_min + 1) * float(self._last_z_step_um))
            try:
                self._updating_z_widgets = True
                self.crop_z.z_range_um.value = float(z_range_um)
            finally:
                self._updating_z_widgets = False
        if show_warning and self._last_sampled_best_z_min is not None:
            if (self._last_sampled_best_z_min < z_min) or (self._last_sampled_best_z_max > z_max):
                self.images_output.value = f"[WARN] slices {z_min}..{z_max} do NOT cover sampled best-zs ({self._last_sampled_best_z_min}..{self._last_sampled_best_z_max})."

    def _apply_z_crop(self, z_range_um: float = 200.0):
        if self._cropped_stack_xy_raw is None:
            self.images_output.value = "[WARN] Create cropped aligned stack first."
            return
        try:
            z_min = int(self.crop_z.z_min.value)
            z_max = int(self.crop_z.z_max.value)
        except Exception:
            z_min = z_max = None
        nz = int(self._cropped_stack_xy_raw.shape[0])
        if z_min is None or z_max is None:
            z_min, z_max = self._compute_z_min_max_slices(z_range_um)
        if z_min is None or z_max is None:
            self.images_output.value = "[WARN] Cannot determine z_min/z_max (need voxel z_step + global_z0)."
            return
        z_min = int(np.clip(z_min, 0, nz - 1))
        z_max = int(np.clip(z_max, 0, nz - 1))
        if z_min > z_max:
            z_min, z_max = z_max, z_min
        cropped_z = self._cropped_stack_xy_raw[z_min : z_max + 1]
        self._cropped_stack_z_raw = cropped_z
        display_stack = self._scale_to_uint8_view(cropped_z)
        self._cropped_layer = self._add_or_update_image_layer(self._cropped_layer, display_stack, "cropped_rotated")
        if self._last_sampled_best_z_min is not None:
            if (self._last_sampled_best_z_min < z_min) or (self._last_sampled_best_z_max > z_max):
                self.images_output.value = f"[WARN] Z crop slices {z_min}..{z_max} does NOT cover sampled best-zs ({self._last_sampled_best_z_min}..{self._last_sampled_best_z_max})."
                return
        self.images_output.value = f"[OK] Z-cropped stack created (slices {z_min}..{z_max})."

    # ---- Rescale ----
    def get_cropped_rotated(self, prefer_z_cropped: bool = True, copy: bool = True):
        arr = None
        if prefer_z_cropped and self._cropped_stack_z_raw is not None:
            arr = self._cropped_stack_z_raw
        elif self._cropped_stack_xy_raw is not None:
            arr = self._cropped_stack_xy_raw
        if arr is None:
            return None
        return np.array(arr, copy=True) if copy else arr

    def get_cropped_rotated_5um(self, copy: bool = True):
        if self._cropped_stack_5um_raw is None:
            return None
        return np.array(self._cropped_stack_5um_raw, copy=True) if copy else self._cropped_stack_5um_raw

    def _rescale_stack_to_target_voxel_um(self, arr: np.ndarray, voxel_um: tuple[float, float, float], target_voxel_um: tuple[float, float, float] = (5.0, 5.0, 5.0)) -> np.ndarray:
        arr = np.asarray(arr)
        if arr.ndim != 3:
            raise ValueError(f"Expected 3D (Z,Y,X); got shape={arr.shape}")
        z_um, y_um, x_um = (float(voxel_um[0]), float(voxel_um[1]), float(voxel_um[2]))
        tz, ty, tx = (float(target_voxel_um[0]), float(target_voxel_um[1]), float(target_voxel_um[2]))
        old_shape = np.array(arr.shape, dtype=float)
        old_spacing = np.array([z_um, y_um, x_um], dtype=float)
        tgt_spacing = np.array([tz, ty, tx], dtype=float)
        new_shape = np.maximum(1, np.rint(old_shape * (old_spacing / tgt_spacing))).astype(int)
        anti_aliasing = bool(np.any(new_shape < old_shape.astype(int)))
        out = resize(arr, tuple(int(x) for x in new_shape.tolist()), order=1, preserve_range=True, anti_aliasing=anti_aliasing)
        return out.astype(arr.dtype, copy=False)

    def _apply_rescale_to_5um(self):
        cropped = self.get_cropped_rotated(prefer_z_cropped=True, copy=True)
        if cropped is None:
            self.images_output.value = "[WARN] No cropped stack to rescale. Do XY/Z crop first."
            return
        voxel_um = self.get_voxel_size_um()
        if any(v is None for v in voxel_um):
            self.images_output.value = "[WARN] Missing voxel metadata (need z_step and xy_step in µm)."
            return
        try:
            self._cropped_stack_5um_raw = self._rescale_stack_to_target_voxel_um(cropped, voxel_um, (5.0, 5.0, 5.0))
        except Exception as e:
            self._cropped_stack_5um_raw = None
            self.images_output.value = f"[ERROR] Rescale failed: {type(e).__name__}: {e}"
            return
        self.images_output.value = f"[OK] Rescaled to 5×5×5 µm. New shape={tuple(np.asarray(self._cropped_stack_5um_raw).shape)}"

    # ---- small helpers used by segmentation ----
    def _signed_orientation(self, region) -> float:
        # region.orientation is in radians; convert to degrees for rotate() usage
        try:
            return float(np.rad2deg(region.orientation))
        except Exception:
            return 0.0

    def _corners_touch_border(self, corners_xy, shape, margin: int = 5) -> bool:
        if corners_xy is None:
            return True
        h, w = int(shape[0]), int(shape[1])
        c = np.asarray(corners_xy, dtype=float)
        x = c[:, 0]
        y = c[:, 1]
        return bool((x.min() < margin) or (y.min() < margin) or (x.max() > (w - 1 - margin)) or (y.max() > (h - 1 - margin)))

    def _oriented_rect_corners_crop_necks_and_flares(self, mask: np.ndarray):
        # Minimal oriented bounding rectangle via regionprops; returns corners in (x,y)
        lbl = label(mask.astype(bool))
        props = regionprops(lbl)
        if not props:
            return None, None, None
        p = max(props, key=lambda r: r.area)
        minr, minc, maxr, maxc = p.bbox
        corners_xy = np.array([[minc, minr], [maxc, minr], [maxc, maxr], [minc, maxr]], dtype=float)
        return corners_xy, 0.0, np.array([(minc + maxc) / 2.0, (minr + maxr) / 2.0], dtype=float)

    def _mask_out_organoid(self, img2d: np.ndarray) -> np.ndarray:
        # Minimal placeholder: no organoid masking by default
        return np.zeros_like(img2d, dtype=bool)

In [3]:
# Export-on-close (kept in DeviceSegmentationApp via method attachment)
# This cell keeps the big class cell untouched while still giving `app.attach_export_on_close()`.


def _vascumap_attach_export_on_close(self, namespace=None):
    """Attach a Napari close handler that exports results.

    If `namespace` is a dict (e.g. globals()), it populates:
      - cropped_rotated_result
      - cropped_rotated_5um
      - cropped_rotated_voxel_um
      - cropped_rotated_outputs
    """

    self._export_namespace = namespace
    self._exported_on_close = False

    def _export(*_args, **_kwargs):
        if getattr(self, "_exported_on_close", False):
            return
        self._exported_on_close = True
        try:
            cropped_rotated_result = self.get_cropped_rotated(prefer_z_cropped=True, copy=True)
            cropped_rotated_5um = self.get_cropped_rotated_5um(copy=True)
            cropped_rotated_voxel_um = tuple(self.get_voxel_size_um())
            cropped_rotated_outputs = {
                "cropped_shape": None if cropped_rotated_result is None else tuple(np.asarray(cropped_rotated_result).shape),
                "cropped_5um_shape": None if cropped_rotated_5um is None else tuple(np.asarray(cropped_rotated_5um).shape),
                "voxel_um": cropped_rotated_voxel_um,
            }

            # Store on the app as well (so it's accessible even if no namespace is provided)
            self.cropped_rotated_result = cropped_rotated_result
            self.cropped_rotated_5um = cropped_rotated_5um
            self.cropped_rotated_voxel_um = cropped_rotated_voxel_um
            self.cropped_rotated_outputs = cropped_rotated_outputs

            if isinstance(self._export_namespace, dict):
                self._export_namespace["cropped_rotated_result"] = cropped_rotated_result
                self._export_namespace["cropped_rotated_5um"] = cropped_rotated_5um
                self._export_namespace["cropped_rotated_voxel_um"] = cropped_rotated_voxel_um
                self._export_namespace["cropped_rotated_outputs"] = cropped_rotated_outputs

            print("[OK] Exported cropped outputs:", cropped_rotated_outputs)
        except Exception as e:
            print(f"[WARN] Failed to export cropped outputs on close: {e}")

    connected = False
    try:
        self.viewer.window.events.closing.connect(_export)
        connected = True
    except Exception:
        pass
    try:
        self.viewer.window._qt_window.destroyed.connect(_export)
        connected = True
    except Exception:
        pass

    if not connected:
        print("[WARN] Could not attach a close handler.")
    return connected


# Attach the method to the class so it behaves like it's part of the app.
DeviceSegmentationApp.attach_export_on_close = _vascumap_attach_export_on_close


In [4]:
# Launch GUI + export results on Napari close
app = DeviceSegmentationApp()
viewer = app.viewer
segment_and_view = app.segment_and_view
list_images = app.list_images
images_output = app.images_output
main_panel = app.main_panel

# These globals get populated when Napari closes.
cropped_rotated_result = None
cropped_rotated_5um = None
cropped_rotated_voxel_um = None
cropped_rotated_outputs = None

app.attach_export_on_close(globals())


True

C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_1533568\1924328847.py:814: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `ProjectiveTransform.from_estimate` class constructor instead.
  if not tform.estimate(dst, c):


[OK] Exported cropped outputs: {'cropped_shape': None, 'cropped_5um_shape': None, 'voxel_um': (7.99810909090909, 1.3000013000013, 1.3000013000013)}
